# ClinHallu complete pipeline
Select a GPU runtime. Extract `clinhallu` into MyDrive and provide your fixed raw splits. The pipeline trains GAER++, B–F ablations, generates the shared five answers, runs the five default-enabled baselines, and builds reports. The eight paid judges are intentionally run one at a time with `--only`; see the README.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/clinhallu
%pip install -r requirements/colab.txt
%pip install -e . --no-deps


## Interruptions are expected

Colab sessions end: the twelve-hour cap, an idle timeout, a reclaimed GPU.
None of it loses your work. Re-run the install cell above and then the same
pipeline cell below, and it continues from where it stopped - including from
the middle of a training epoch.

Because the run directory is on Drive, it outlives the VM. That is why the
first cell mounts Drive and `cd`s into it before anything else.

See `docs/resume.md` for the full picture.

## What is already finished?

Run this before committing a session to see what a re-run would still do.

In [ ]:
!clinhallu status

If installation requests a restart, restart now, then mount Drive and change directory again. Put `train.jsonl`, `val.jsonl`, and `eval_data.jsonl` under `data/raw/`, and set the audited row counts before running.

In [ ]:
import os
import getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')


In [ ]:
!clinhallu validate --config configs/conditions/c5.yaml
!clinhallu generate --config configs/generation/shared_five.yaml --dry-run
!clinhallu pipeline --condition c5 --seed 13 --dry-run


## Run the pipeline

Re-run this exact cell after any interruption. Finished stages are skipped.

In [ ]:
!clinhallu pipeline --condition c5 --seed 13


### Forcing a stage to run again

```
!clinhallu pipeline --condition c5 --seed 13 --force-stage baselines
!clinhallu pipeline --condition c5 --seed 13 --no-resume
```

## Individual shared-answer commands
Use these if running the two consistency baselines separately. Completed generation requests are cached. Five answers come from five independent completions per unique question/context. Both scoring commands reuse the same answer file, but use distinct local NLI scorers. Default generation is evaluation only; see `docs/shared_answer_protocol.md`.

In [ ]:
!clinhallu generate --config configs/generation/shared_five.yaml


In [ ]:
!clinhallu baseline run --only selfcheckgpt
!clinhallu baseline run --only self_consistency


In [ ]:
!clinhallu report build
